In [9]:
import pandas as pd

# Load the survey dataset

In [10]:
import pandas as pd

survey = pd.read_csv("../processed_data/survey_cleaned.csv")

# Create the average previous period length

In [11]:
survey["Average_Previous_Period_Length"] = (
    survey[
        [
            "Prev_1_Period_Length",
            "Prev_2_Period_Length",
            "Prev_3_Period_Length"
        ]
    ].mean(axis=1)
)

# Create the cycle prediction features

In [12]:
cycle_features = [
    "Age",
    "BMI",
    "Age_At_Menarche",
    "Prev_1_Cycle_Length",
    "Prev_2_Cycle_Length",
    "Prev_3_Cycle_Length",
    "Prev_1_Period_Length",
    "Prev_2_Period_Length",
    "Prev_3_Period_Length",
    "Sleep_Hours",
    "Stress_Level",
    "Exercise_Frequency",
    "Medication_Contraceptive"
]

# Rebuild the survey feature table with categorical columns encoded numerically
survey = pd.read_csv("../processed_data/survey_cleaned.csv")
survey["Average_Previous_Period_Length"] = (
    survey[["Prev_1_Period_Length", "Prev_2_Period_Length", "Prev_3_Period_Length"]].mean(axis=1)
)

survey["Stress_Level"] = (
    survey["Stress_Level"]
    .map({"Very Low": 1, "Low": 2, "Moderate": 3, "High": 4, "Very High": 5})
    .fillna(0)
    .astype(float)
)

survey["Exercise_Frequency"] = (
    survey["Exercise_Frequency"]
    .map({"Never": 0, "1-2 days/week": 1, "3-4 days/week": 2, "5+ days/week": 2})
    .fillna(0)
    .astype(float)
)

survey["Medication_Contraceptive"] = (
    survey["Medication_Contraceptive"]
    .map({"No": 0, "Yes": 1, "Not Sure": 0})
    .fillna(0)
    .astype(float)
)

X_cycle_survey = survey[cycle_features]

# Load the preprocessing pipeline

In [14]:
import joblib

cycle_pipeline = joblib.load(
    "../models/cycle_preprocessing_pipeline.pkl"
)


# Transform the survey data

In [16]:
survey = pd.read_csv("../processed_data/survey_cleaned.csv")
survey["Average_Previous_Period_Length"] = (
    survey[
        [
            "Prev_1_Period_Length",
            "Prev_2_Period_Length",
            "Prev_3_Period_Length"
        ]
    ].mean(axis=1)
)

survey["Stress_Level"] = (
    survey["Stress_Level"]
    .map({"Very Low": 1, "Low": 2, "Moderate": 3, "High": 4, "Very High": 5})
    .fillna(0)
    .astype(float)
)

survey["Exercise_Frequency"] = (
    survey["Exercise_Frequency"]
    .map({"Never": 0, "1-2 days/week": 1, "3-4 days/week": 2, "5+ days/week": 2})
    .fillna(0)
    .astype(float)
)

survey["Medication_Contraceptive"] = (
    survey["Medication_Contraceptive"]
    .map({"No": 0, "Yes": 1, "Not Sure": 0})
    .fillna(0)
    .astype(float)
)

X_cycle_survey = survey[cycle_features]
X_cycle_survey_processed = cycle_pipeline.transform(X_cycle_survey)

# Load the best cycle model

In [18]:
cycle_model = joblib.load(
    "../models/trained_models/Linear_Regression_cycle.pkl"
)


# Predict cycle length

In [19]:
survey["Predicted_Next_Cycle_Length"] = cycle_model.predict(
    X_cycle_survey_processed
)

# Predict period length

In [20]:
X_period_survey = survey[
    ["Average_Previous_Period_Length"]
]


# scale

In [22]:
period_scaler = joblib.load(
    "../models/period_scaler.pkl"
)

X_period_survey = survey[["Average_Previous_Period_Length"]]
X_period_survey = period_scaler.transform(
    X_period_survey
)


# load best period model

In [24]:
period_model = joblib.load(
    "../models/trained_models/Linear_Regression_period.pkl"
)


# Predict:

In [25]:
survey["Predicted_Next_Period_Length"] = period_model.predict(
    X_period_survey
)

# Save the predictions

In [26]:
from pathlib import Path

output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

survey.to_csv(
    output_dir / "survey_predictions.csv",
    index=False
)
